Document Upload

In [1]:
from langchain_community.document_loaders import PyPDFLoader

files = [
    "Donald John Trump.pdf",
    "Narendra Damodardas Modi.pdf"
]

documents = []

for file in files:
    loader = PyPDFLoader(file)
    docs = loader.load()
    documents.extend(docs)

In [2]:
len(documents)

2

In [3]:
documents[1].metadata

{'producer': 'Microsoft® Word 2021',
 'creator': 'Microsoft® Word 2021',
 'creationdate': '2026-03-14T00:09:59+05:30',
 'author': 'Harshat Mandial',
 'moddate': '2026-03-14T00:09:59+05:30',
 'source': 'Narendra Damodardas Modi.pdf',
 'total_pages': 1,
 'page': 0,
 'page_label': '1'}

In [4]:
import fitz

pdf1 = fitz.open("Donald John Trump.pdf")
pdf2 = fitz.open("Narendra Damodardas Modi.pdf")

print("PDF1 pages:", len(pdf1))
print("PDF2 pages:", len(pdf2))

PDF1 pages: 1
PDF2 pages: 1


In [5]:
import fitz
from tqdm.auto import tqdm
import os

def text_formatter(t: str) -> str:
    """Clean extracted text."""
    return t.replace("\n", " ").strip()


def open_and_read_pdf(documents):
    """Read multiple PDFs and return pages with metadata."""
    
    pages_and_texts = []

    for pdf_path in documents:

        doc = fitz.open(pdf_path)

        for page_number, page in tqdm(enumerate(doc), total=len(doc)):

            t = page.get_text()
            t = text_formatter(t)

            pages_and_texts.append({
                "source": os.path.basename(pdf_path),
                "page_number": page_number,
                "page_char_count": len(t),
                "page_word_count": len(t.split()),
                "page_sentence_count": len(t.split(". ")),
                "page_token_count": len(t) / 4,
                "text": t
            })

    return pages_and_texts

In [6]:
documents = [
    "Donald John Trump.pdf",
    "Narendra Damodardas Modi.pdf"
]

texts = open_and_read_pdf(documents)

print(len(texts))
print(texts[:2])

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

2
[{'source': 'Donald John Trump.pdf', 'page_number': 0, 'page_char_count': 3746, 'page_word_count': 567, 'page_sentence_count': 29, 'page_token_count': 936.5, 'text': "Donald John Trump  Donald John Trump (born June 14, 1946) is an American politician, media personality, and  businessman who is the 47th president of the United States. A member of the Republican Party, he  served as the 45th president from 2017 to 2021.  Born into a wealthy New York City family, Trump graduated from the University of Pennsylvania in  1968 with a bachelor's degree in economics. He became the president of his family's real estate  business in 1971, renamed it the Trump Organization, and began acquiring and building skyscrapers,  hotels, casinos, and golf courses. He launched side ventures, many licensing the Trump name, and  filed for six business bankruptcies in the 1990s and 2000s. From 2004 to 2015, he hosted the reality  television show The Apprentice, bolstering his fame as a billionaire. Presenting

In [7]:
from langchain_core.documents import Document

In [8]:
docs = [
    Document(
        page_content=item["text"],
        metadata={
            "source": item["source"],
            "page": item["page_number"]
        }
    )
    for item in texts
]

In [9]:
print(docs[1])

page_content='Narendra Damodardas Modi  Narendra Damodardas Modi(born 17 September 1950) is an Indian politician who has served as  the prime minister of India since 2014. Modi was the chief minister of Gujarat from 2001 to 2014 and  is the member of parliament (MP) for Varanasi. He is a member of the Bharatiya Janata Party (BJP)  and of the Rashtriya Swayamsevak Sangh (RSS), a right-wing Hindutva paramilitary volunteer  organisation. He is the longest-serving prime minister outside the Indian National Congress.  Modi was born and raised in Vadnagar, Bombay State (present-day Gujarat), where he completed his  secondary education. He was introduced to the RSS at the age of eight, becoming a full-time worker  for the organisation in Gujarat in 1971. The RSS assigned him to the BJP in 1985, and he rose through  the party hierarchy, becoming general secretary in 1998.[b] In 2001, Modi was appointed chief  minister of Gujarat and elected to the legislative assembly soon after. His administr

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n","\n"," "],
    chunk_size=200,
    chunk_overlap=0
)

chunks = splitter.split_documents(docs)
len(chunks)

40

In [11]:
print(len(chunks))
print(chunks[0])

40
page_content='Donald John Trump  Donald John Trump (born June 14, 1946) is an American politician, media personality, and  businessman who is the 47th president of the United States. A member of the Republican' metadata={'source': 'Donald John Trump.pdf', 'page': 0}


In [12]:
# Numner of character
for chunk in chunks:
    print(len(chunk.page_content))

195
190
198
196
192
193
194
199
193
199
198
194
199
195
198
198
194
197
197
8
196
189
197
199
191
193
197
191
191
195
192
199
199
184
198
192
195
195
196
179


In [13]:
# Number of words
for chunk in chunks:
    print(len(chunk.page_content.split()))

32
33
29
32
27
28
29
32
32
33
31
29
31
25
28
32
27
27
29
1
31
33
26
36
30
29
27
27
26
30
24
29
29
26
29
29
25
25
23
26


In [14]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-mpnet-base-v2")

C:\Users\Harshit\anaconda3\envs\honeyenv\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\Harshit\anaconda3\envs\honeyenv\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [15]:
chunk_embeddings = []

for chunk in chunks:
    
    text = chunk.page_content
    
    embedding = embedding_model.encode(text)

    chunk_embeddings.append({
        "text": text,
        "source": chunk.metadata["source"],
        "page": chunk.metadata["page"],
        "embedding": embedding
    })

len(chunk_embeddings)

40

In [16]:
chunk_embeddings[0]

{'text': 'Donald John Trump  Donald John Trump (born June 14, 1946) is an American politician, media personality, and  businessman who is the 47th president of the United States. A member of the Republican',
 'source': 'Donald John Trump.pdf',
 'page': 0,
 'embedding': array([-4.24477123e-02, -1.26401652e-02,  1.79212801e-02,  8.33012387e-02,
         1.31209670e-02, -3.46341878e-02, -6.21315874e-02,  5.94581589e-02,
         6.68126419e-02, -2.20572986e-02,  4.85998429e-02,  6.45474298e-03,
        -1.67284850e-02, -4.73862216e-02,  1.81190781e-02, -5.70763908e-02,
         3.16729285e-02, -2.53756326e-02,  6.72133267e-02, -3.65357213e-02,
        -5.66644706e-02, -7.84789585e-03, -2.38640085e-02, -1.09962067e-02,
        -4.26464006e-02, -4.15838063e-02,  2.21085120e-02,  1.30016357e-02,
        -1.88923310e-02,  3.78069729e-02, -3.44792716e-02,  3.17006707e-02,
         7.18462281e-03,  2.66238283e-02,  2.29169768e-06,  6.19543251e-03,
         5.55633986e-03,  3.66981584e-03,  3.70

Connecting Jupyter to Postgresql

In [17]:
import psycopg2

conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="1234",
    host="localhost",
    port="5432"
)

cursor = conn.cursor()

print("Connected to PostgreSQL")

Connected to PostgreSQL


In [18]:
try:
    for item in chunk_embeddings:
        cursor.execute(
            """
            INSERT INTO rag_chunks (source, page, chunk_text, embedding)
            VALUES (%s, %s, %s, %s)
            """,
            (
                item["source"],
                item["page"],
                item["text"],
                item["embedding"].tolist()
            )
        )
    conn.commit()
    print("Data inserted successfully")

except Exception as e:
    conn.rollback()  # Rollback transaction to clear error state
    print("Error occurred:", e)

Data inserted successfully


In [19]:
cursor.execute("SELECT current_database();")
print(cursor.fetchone())

('postgres',)


Retrieval And Generation

In [20]:
query = "Who is Narendra Modi?"

query_embedding = embedding_model.encode(query)

In [21]:
conn.rollback()

In [22]:
cursor.execute(
    """
    SELECT chunk_text, source
    FROM rag_chunks
    ORDER BY embedding <-> %s::vector
    LIMIT 3
    """,
    (query_embedding.tolist(),)
)
results = cursor.fetchall()

results

[('Narendra Damodardas Modi  Narendra Damodardas Modi(born 17 September 1950) is an Indian politician who has served as  the prime minister of India since 2014. Modi was the chief minister of Gujarat',
  'Narendra Damodardas Modi.pdf'),
 ('Narendra Damodardas Modi  Narendra Damodardas Modi(born 17 September 1950) is an Indian politician who has served as  the prime minister of India since 2014. Modi was the chief minister of Gujarat',
  'Narendra Damodardas Modi.pdf'),
 ('Narendra Damodardas Modi  Narendra Damodardas Modi(born 17 September 1950) is an Indian politician who has served as  the prime minister of India since 2014. Modi was the chief minister of Gujarat',
  'Narendra Damodardas Modi.pdf')]

In [23]:
results = cursor.fetchall()

In [24]:
def build_prompt(context, question):

    return f"""
You are a helpful AI assistant.

Use ONLY the context below to answer the question.

If the answer is not present, say: "I don't know."

Context:
{context}

Question:
{question}

Answer in 2-3 sentences.
Also mention the source document name.
"""

In [25]:
import ollama

def generate_answer(prompt):

    response = ollama.chat(
        model="llama3.2:1b",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response["message"]["content"]

In [26]:
def rag_pipeline(question):
    try:
        # Step 1: Get embedding
        query_embedding = embedding_model.encode(question)

        # Step 2: Retrieve chunks
        cursor.execute(
            """
            SELECT chunk_text, source
            FROM rag_chunks
            ORDER BY embedding <=> (%s)::vector
            LIMIT 3
            """,
            (query_embedding.tolist(),)
        )

        results = cursor.fetchall()

        # Step 3: Build context
        context = ""
        sources = set()
        for row in results:
            context += row[0] + "\n\n"
            sources.add(row[1])

        # Step 4: Prompt
        prompt = build_prompt(context, question)

        # Step 5: LLM (Llama)
        answer = generate_answer(prompt)

        return answer, sources

    except Exception as e:
        # Reset transaction if anything goes wrong
        conn.rollback()
        print("Error:", e)
        return None, None

In [29]:
question = "Fullname of Modi?"

answer, sources = rag_pipeline(question)

print("Answer:\n", answer)
print("\nSources:\n", sources)

Answer:
 Narendra Damodardas Modi

Sources:
 {'Narendra Damodardas Modi.pdf'}
